# 04 — Fine-tuning FinBERT + Comparaison finale + Interface Gradio

Ce notebook :
1. Fine-tune **FinBERT** (`ProsusAI/finbert`) sur Financial PhraseBank via HuggingFace 🤗
2. Évalue le modèle (classification report + matrice de confusion)
3. Produit un **tableau comparatif** de tous les modèles
4. Lance une **interface Gradio** pour tester en live

> GPU fortement recommandé (Google Colab / Kaggle). Sur CPU, réduire `NUM_EPOCHS` à 1.

In [ ]:
# Installations si nécessaire (décommenter)
# !pip install transformers datasets accelerate gradio -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json, os
from pathlib import Path

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score
)

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 110
SEED = 42

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')
print(f'Transformers : {__import__("transformers").__version__}')

## 1. Chargement des données

In [ ]:
df = pd.read_csv(
    '../data/all-data.csv',
    encoding='latin-1',
    header=None,
    names=['sentiment', 'headline']
)

# FinBERT a été pré-entraîné avec les labels : positive=0, negative=1, neutral=2
# On réutilise le même mapping pour être compatible avec les poids pré-entraînés
LABEL2ID = {'positive': 0, 'negative': 1, 'neutral': 2}
ID2LABEL  = {v: k for k, v in LABEL2ID.items()}

df['label'] = df['sentiment'].map(LABEL2ID)
print(f'Shape : {df.shape}')
print(df['sentiment'].value_counts())
df.head()

## 2. Split & création du Dataset HuggingFace

In [ ]:
train_df, test_df = train_test_split(
    df[['headline', 'label']], test_size=0.2,
    random_state=SEED, stratify=df['label']
)
train_df, val_df = train_test_split(
    train_df, test_size=0.1,
    random_state=SEED, stratify=train_df['label']
)

print(f'Train : {len(train_df)} | Val : {len(val_df)} | Test : {len(test_df)}')

dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df.reset_index(drop=True)),
    'val':   Dataset.from_pandas(val_df.reset_index(drop=True)),
    'test':  Dataset.from_pandas(test_df.reset_index(drop=True))
})
dataset

## 3. Tokenisation FinBERT

In [ ]:
MODEL_NAME = 'ProsusAI/finbert'
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch['headline'],
        truncation=True,
        padding='max_length',
        max_length=MAX_LENGTH
    )

tokenized = dataset.map(tokenize, batched=True)
tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
print(tokenized)

## 4. Chargement du modèle FinBERT

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID
)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Paramètres totaux    : {total_params:,}')
print(f'Paramètres entraînés : {trainable:,}')

## 5. Fine-tuning avec HuggingFace Trainer

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_macro': f1_score(labels, preds, average='macro')
    }

NUM_EPOCHS  = 5
BATCH_SIZE  = 16
OUTPUT_DIR  = '../results/finbert_finetuned'

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    logging_dir=f'{OUTPUT_DIR}/logs',
    logging_steps=50,
    fp16=(device == 'cuda'),
    seed=SEED,
    report_to='none'   # désactive W&B / MLflow
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['val'],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()

## 6. Évaluation sur le test set

In [ ]:
preds_output = trainer.predict(tokenized['test'])
y_pred_fb    = np.argmax(preds_output.predictions, axis=-1)
y_true_fb    = preds_output.label_ids

acc_finbert  = accuracy_score(y_true_fb, y_pred_fb)
class_names  = [ID2LABEL[i] for i in sorted(ID2LABEL)]

print(f'Accuracy FinBERT : {acc_finbert:.4f}')
print('\n--- Classification Report ---')
print(classification_report(y_true_fb, y_pred_fb, target_names=class_names))

In [ ]:
cm     = confusion_matrix(y_true_fb, y_pred_fb)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True)
annot  = np.array([[f'{v}\n({p:.0%})' for v, p in zip(r_v, r_p)]
                   for r_v, r_p in zip(cm, cm_pct)])

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=annot, fmt='', cmap='Purples',
    xticklabels=class_names, yticklabels=class_names,
    linewidths=0.5, ax=ax
)
ax.set_title(f'Matrice de confusion FinBERT\nAcc = {acc_finbert:.3f}',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Prédit')
ax.set_ylabel('Réel')
plt.tight_layout()
plt.show()

## 7. Tableau comparatif final

In [ ]:
# Chargement des scores précédents (notebooks 02 & 03)
scores_path = Path('../results/baseline_scores.json')
if scores_path.exists():
    with open(scores_path) as f:
        prev_scores = json.load(f)
else:
    # Valeurs de secours si les notebooks précédents n'ont pas été exécutés
    prev_scores = {'LogisticRegression': None, 'NaiveBayes': None, 'LSTM': None}

comparison = pd.DataFrame([
    {
        'Modèle'          : 'TF-IDF + Logistic Regression',
        'Type'            : 'Baseline',
        'Accuracy Test'   : prev_scores.get('LogisticRegression'),
        'Paramètres'      : '~',
        'Avantages'       : 'Rapide, interprétable',
        'Inconvénients'   : 'Ignore le contexte'
    },
    {
        'Modèle'          : 'TF-IDF + Naive Bayes',
        'Type'            : 'Baseline',
        'Accuracy Test'   : prev_scores.get('NaiveBayes'),
        'Paramètres'      : '~',
        'Avantages'       : 'Très rapide, peu de données',
        'Inconvénients'   : 'Hypothèse d\'indépendance'
    },
    {
        'Modèle'          : 'Bi-LSTM (Embedding trainable)',
        'Type'            : 'Deep Learning',
        'Accuracy Test'   : prev_scores.get('LSTM'),
        'Paramètres'      : '~1.5M',
        'Avantages'       : 'Capture séquences, contexte local',
        'Inconvénients'   : 'Pas de pré-entraînement financier'
    },
    {
        'Modèle'          : 'FinBERT (fine-tuned)',
        'Type'            : 'Transformer',
        'Accuracy Test'   : round(acc_finbert, 4),
        'Paramètres'      : '~110M',
        'Avantages'       : 'SOTA, pré-entraîné domaine financier',
        'Inconvénients'   : 'Lourd, coût GPU'
    }
])

print('='*70)
print('       TABLEAU COMPARATIF — SENTIMENT ANALYSIS FINANCIÈRE')
print('='*70)
print(comparison[['Modèle', 'Type', 'Accuracy Test']].to_string(index=False))
print('='*70)
comparison

In [ ]:
# Visualisation graphique du comparatif
valid = comparison.dropna(subset=['Accuracy Test'])
colors = ['#3498db', '#3498db', '#e67e22', '#9b59b6']

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(
    valid['Modèle'], valid['Accuracy Test'].astype(float),
    color=colors[:len(valid)], edgecolor='white', height=0.5
)
for bar, val in zip(bars, valid['Accuracy Test']):
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
            f'{float(val):.3f}', va='center', fontweight='bold', fontsize=11)
ax.set_xlim(0, 1.1)
ax.set_xlabel('Accuracy (Test set)', fontsize=11)
ax.set_title('Comparaison des modèles — Financial Sentiment Analysis',
             fontsize=12, fontweight='bold')
ax.axvline(x=0.5, color='gray', linestyle='--', lw=0.8, label='Baseline aléatoire')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 8. Interface Gradio

In [ ]:
import gradio as gr

# Pipeline d'inférence
from transformers import pipeline as hf_pipeline

sentiment_pipeline = hf_pipeline(
    'text-classification',
    model=model,
    tokenizer=tokenizer,
    device=0 if device == 'cuda' else -1,
    return_all_scores=True
)

EMOJI = {'positive': '📈', 'negative': '📉', 'neutral': '➡️'}

def predict_gradio(text: str) -> dict:
    if not text.strip():
        return {'error': 'Veuillez entrer une phrase.'}
    results = sentiment_pipeline(text[:512])[0]
    scores  = {EMOJI[r['label']] + ' ' + r['label']: round(r['score'], 4)
               for r in results}
    return scores

examples = [
    "The company announced record profits and raised its dividend.",
    "CEO resigned amid accounting fraud scandal sending shares tumbling 30%.",
    "The board approved a share buyback program worth $500 million.",
    "Revenue remained flat compared to the previous quarter.",
    "Acquisition talks with a major competitor have reportedly collapsed."
]

with gr.Blocks(title='Financial Sentiment Analysis', theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 📊 Financial Sentiment Analysis — FinBERT
    Analyse le sentiment d'un titre de news financière avec un modèle FinBERT fine-tuné.
    """)

    with gr.Row():
        with gr.Column(scale=2):
            inp = gr.Textbox(
                label='News headline',
                placeholder='Enter a financial news headline…',
                lines=3
            )
            btn = gr.Button('Analyser', variant='primary')

        with gr.Column(scale=1):
            out = gr.Label(label='Scores de sentiment')

    gr.Examples(examples=examples, inputs=inp)

    btn.click(fn=predict_gradio, inputs=inp, outputs=out)
    inp.submit(fn=predict_gradio, inputs=inp, outputs=out)

demo.launch(share=False)

## 9. Sauvegarde du modèle fine-tuné

In [ ]:
save_dir = Path('../results/finbert_finetuned/best_model')
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print(f'Modèle sauvegardé dans : {save_dir}')

# Mise à jour du fichier de scores
scores = {}
if scores_path.exists():
    with open(scores_path) as f:
        scores = json.load(f)
scores['FinBERT'] = round(float(acc_finbert), 4)
with open(scores_path, 'w') as f:
    json.dump(scores, f, indent=2)
print('Scores finaux :', scores)